# TGN on Kaggle

Trains a TGN of EarlyMule-TG (`python -m earlymule.models.tgn --config CONFIG`) on the exported dataset `80K-earlymule-mule`, the seeds in `SEEDS` in parallel on the two T4 GPUs (GPU = seed % 2). `tgn.yaml` is the minimal TGN; `tgn_features.yaml` also gives the account head the daily LightGBM features of each row. Each run writes validation and test scores and `run_meta.json` to `/kaggle/working/<config name>/seed<N>/`; evaluation runs locally on these files.

Inputs: the private datasets `verzol/earlymule-80k-earlymule-mule` and `verzol/earlymule-10k-earlymule-mule` (made by `python -m earlymule.data.export_events`). The environment check of version 1 lives in `python -m earlymule.models.tgn_smoke`.

Push and run from the repository root: `kaggle kernels push -p notebooks/kaggle/tgn_spike`.

In [ ]:
# Run parameters: model config, seeds, learning rates to sweep (None = the config's), commit (None = latest main),
# and how many runs at once (each loads the row features, so RAM bounds it)
CONFIG = 'tgn.yaml'
SEEDS = [0, 1, 2]
LRS = [None]
COMMIT = None
MAX_PARALLEL = 3

In [ ]:
# GPU and driver
!nvidia-smi

In [ ]:
# Clone the repository at COMMIT and install it with PyG (torch is preinstalled on Kaggle)
!git clone -q https://github.com/Verzol/earlymule-tg.git /tmp/earlymule-tg
if COMMIT:
    !git -C /tmp/earlymule-tg checkout -q {COMMIT}
!git -C /tmp/earlymule-tg log -1 --format='%h %s'
!pip install -q -e /tmp/earlymule-tg --no-deps
!pip install -q torch_geometric

The dataset folder is found by its manifest, wherever Kaggle mounts it. Every (learning rate, seed) pair runs in its own process, alternating between the two GPUs, at most `MAX_PARALLEL` at a time; outputs go to `/kaggle/working/<config name>/seed<N>[_lr<X>]/`, logs next to them.

In [ ]:
# Train every (learning rate, seed) pair, MAX_PARALLEL at a time on two GPUs
import json, os, subprocess
from pathlib import Path
data = next(m.parent for m in Path('/kaggle/input').rglob('manifest.json')
            if json.loads(m.read_text())['dataset'] == '80K-earlymule-mule')
out = Path('/kaggle/working') / Path(CONFIG).stem
out.mkdir(parents=True, exist_ok=True)
jobs = [(seed, lr) for lr in LRS for seed in SEEDS]
codes = []
for start in range(0, len(jobs), MAX_PARALLEL):
    procs = []
    for i, (seed, lr) in enumerate(jobs[start:start + MAX_PARALLEL]):
        name = f'seed{seed}' if lr is None else f'seed{seed}_lr{lr:g}'
        env = {**os.environ, 'CUDA_VISIBLE_DEVICES': str((start + i) % 2)}
        cmd = ['python', '-m', 'earlymule.models.tgn', '--data', str(data), '--config', CONFIG, '--seed', str(seed),
               '--out', str(out / name)] + ([] if lr is None else ['--lr', str(lr)])
        procs.append(subprocess.Popen(cmd, cwd='/tmp/earlymule-tg', env=env,
                                      stdout=open(out / f'{name}.log', 'w'), stderr=subprocess.STDOUT))
    codes += [p.wait() for p in procs]
print(codes)

In [ ]:
# Logs and run summaries
for log in sorted(out.glob('*.log')):
    print(f'--- {log.stem}')
    print(log.read_text()[-3000:])